# AI 판사의 음주운전 형량 판별기   
### Qwen/Qwen2.5-3B-Instruct 버전

| 항목 | 내용 |
|------|------|
| 입력 | GitHub에서 자동 다운로드 (법정형.docx · 양형기준표.docx · 음주운전_판결문_추출 v5.csv) |
| 모델 | `Qwen/Qwen2.5-3B-Instruct` (HuggingFace Transformers) |
| 출력 | 건별 콘솔 출력 — 법원 / 사건 / 주문 / 범죄사실 / 양형이유 / 실제판결 / AI판사 예측 |

> **런타임 설정** : 상단 메뉴 → **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 실행하세요.

## 셀 1 · 패키지 설치

In [1]:
!pip install -q requests python-docx transformers torch accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.1 MB/s eta 0:00:00


## 셀 2 · 라이브러리 임포트

In [2]:
import csv
import io
import re
import tempfile
from collections import defaultdict
from urllib.parse import quote

import requests
from docx import Document
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU 사용 가능: True
GPU: Tesla T4


## 셀 3 · 설정

> **GitHub 정보는 Public **

In [3]:
# ── GitHub 저장소 정보 ──────────────────────────────────────────
GITHUB_USER   = "ssc-lexdev"   # ← GitHub 사용자명
GITHUB_REPO   = "ssc_project"  # ← 저장소 이름
GITHUB_BRANCH = "main"         # ← 브랜치 이름
GITHUB_FOLDER = ""             # ← 하위 폴더 (루트면 "" 그대로)

# ── GitHub에 올린 파일명  ─────────────
FILE_법정형   = "음주운전 법정형.docx"
FILE_양형기준 = "음주운전 양형기준표.docx"
FILE_판결문   = "음주운전_판결문_추출 v5 - data.csv"

# ── v5-data.csv 열 이름  ─────────────
COL_법원     = "법원"
COL_사건     = "사건"
COL_주문     = "주문"
COL_범죄사실 = "범죄사실"   # ← 모델 입력에 사용
COL_양형이유 = "양형이유"   # ← 모델 입력에 사용

# ── 모델 ID ──────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# ── 내부 상수 ────────────────────────────────────────────────────
형종_순위 = {"징역": 3, "집행유예": 2, "벌금": 1}
SEP       = "=" * 60
SEP2      = "-" * 60

print("설정 완료")

설정 완료


## 셀 4 · GitHub 다운로드 함수

In [4]:
def _gh_url(filename: str) -> str:
    """한글·공백 파일명을 URL 인코딩하여 GitHub raw URL 반환"""
    base = (
        f"https://raw.githubusercontent.com"
        f"/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}"
    )
    if GITHUB_FOLDER:
        return f"{base}/{quote(GITHUB_FOLDER)}/{quote(filename)}"
    return f"{base}/{quote(filename)}"


def download_bytes(filename: str) -> bytes:
    url = _gh_url(filename)
    print(f"  다운로드: {url}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    return r.content


def download_docx_text(filename: str) -> str:
    """GitHub → .docx → 순수 텍스트"""
    data = download_bytes(filename)
    with tempfile.NamedTemporaryFile(suffix=".docx", delete=False) as tmp:
        tmp.write(data)
        tmp_path = tmp.name
    doc = Document(tmp_path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())


def download_csv_rows(filename: str):
    """GitHub → CSV → (fieldnames, rows)  ※ 인코딩 자동 감지"""
    data = download_bytes(filename)
    for enc in ("utf-8-sig", "utf-8", "cp949", "euc-kr"):
        try:
            text = data.decode(enc)
            reader = csv.DictReader(io.StringIO(text))
            fieldnames = list(reader.fieldnames)
            rows = list(reader)
            print(f"  인코딩 감지: {enc}")
            return fieldnames, rows
        except (UnicodeDecodeError, TypeError):
            continue
    raise ValueError("CSV 파일 인코딩을 자동으로 감지할 수 없습니다.")


print("다운로드 함수 정의 완료")

다운로드 함수 정의 완료


## 셀 5 · 유틸리티 함수 (숫자 변환 · 형량 비교 · 성능 평가)

In [5]:
# ── 텍스트 → 숫자 ──────────────────────────────────────────────

def text_to_months(text: str):
    text = text.strip()
    if not text or "만 원" in text or "만원" in text:
        return None
    months = 0.0
    m = re.search(r"(\d+)\s*년", text)
    if m:
        months += int(m.group(1)) * 12
    m = re.search(r"(\d+)\s*개월", text)
    if m:
        months += int(m.group(1))
    return months if months > 0 else None


def text_to_manwon(text: str):
    text = text.strip()
    m = re.search(r"([\d,]+)\s*만", text)
    if m:
        return float(m.group(1).replace(",", ""))
    return None


def label2_to_text(label1: str, label2: str) -> str:
    """Label2 숫자 → 읽기 쉬운 형량 텍스트 (실제 판결 표시용)"""
    try:
        val = float(label2)
    except (ValueError, TypeError):
        return label2
    if label1 in ("징역", "집행유예"):
        total_months = round(val * 12)
        years, months = divmod(total_months, 12)
        if years > 0 and months > 0:
            return f"{years}년 {months}개월"
        elif years > 0:
            return f"{years}년"
        else:
            return f"{months}개월"
    if label1 == "벌금":
        return f"{int(round(val / 10000))}만 원"
    return label2


def label2_to_numeric(label1: str, label2: str):
    """Label2 숫자 → 개월 또는 만 원 (수치 비교용)"""
    try:
        val = float(label2)
    except (ValueError, TypeError):
        return None
    if label1 in ("징역", "집행유예"):
        return val * 12
    if label1 == "벌금":
        return val / 10000
    return None


def normalize_형량(형종: str, 형량_text: str) -> str:
    """AI 출력 형량 텍스트를 정규화 → 실제값과 비교 가능하게"""
    if not 형종 or not 형량_text:
        return ""
    if 형종 == "벌금":
        val = text_to_manwon(형량_text)
        return f"{int(val)}만 원" if val is not None else 형량_text.strip()
    months = text_to_months(형량_text)
    if months is None:
        return 형량_text.strip()
    years, m = divmod(round(months), 12)
    if years > 0 and m > 0:
        return f"{years}년 {m}개월"
    elif years > 0:
        return f"{years}년"
    else:
        return f"{m}개월"


# ── 형량 비교 ──────────────────────────────────────────────────

def compare_sentences(actual_종, actual_량_num, ai_종, ai_량_num) -> str:
    if not actual_종 or not ai_종:
        return "비교불가"
    r_actual = 형종_순위.get(actual_종, 0)
    r_ai     = 형종_순위.get(ai_종, 0)
    if r_actual != r_ai:
        return "실제판결 더 높음" if r_actual > r_ai else "AI판결 더 높음"
    if actual_량_num is None or ai_량_num is None:
        return "형종 동일 (형량 비교불가)"
    if actual_종 in ("징역", "집행유예"):
        tolerance = 12    # 1년 = 12개월 미만 차이는 동일 처리
    elif actual_종 == "벌금":
        tolerance = 500   # 500만 원 미만 차이는 동일 처리
    else:
        tolerance = 0.01
    if abs(actual_량_num - ai_량_num) < tolerance:
        return "동일"
    return "실제판결 더 높음" if actual_량_num > ai_량_num else "AI판결 더 높음"


# ── Classification Report + Confusion Matrix 출력 ─────────────

def print_report(title: str, y_true: list, y_pred: list):
    labels = sorted(set(y_true + y_pred) - {""})
    if not labels:
        print(f"===== {title} =====\n  (데이터 없음)\n")
        return

    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    correct = 0; valid = 0
    for t, p in zip(y_true, y_pred):
        if not t or not p:
            continue
        valid += 1
        if t == p:
            tp[t] += 1; correct += 1
        else:
            fp[p] += 1; fn[t] += 1

    accuracy  = correct / valid if valid else 0
    per_class = {}
    for lbl in labels:
        prec = tp[lbl] / (tp[lbl] + fp[lbl]) if (tp[lbl] + fp[lbl]) > 0 else 0
        rec  = tp[lbl] / (tp[lbl] + fn[lbl]) if (tp[lbl] + fn[lbl]) > 0 else 0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        sup  = sum(1 for t in y_true if t == lbl)
        per_class[lbl] = {"precision": prec, "recall": rec, "f1": f1, "support": sup}

    total_sup = sum(v["support"] for v in per_class.values())
    macro_p = sum(v["precision"] for v in per_class.values()) / len(per_class)
    macro_r = sum(v["recall"]    for v in per_class.values()) / len(per_class)
    macro_f = sum(v["f1"]        for v in per_class.values()) / len(per_class)
    w_p = sum(v["precision"] * v["support"] for v in per_class.values()) / total_sup if total_sup else 0
    w_r = sum(v["recall"]    * v["support"] for v in per_class.values()) / total_sup if total_sup else 0
    w_f = sum(v["f1"]        * v["support"] for v in per_class.values()) / total_sup if total_sup else 0

    W = max(max(len(l) for l in labels), len("weighted avg"))

    print(f"===== {title} =====")
    print(f"  {'':{W}}   {'precision':>9}   {'recall':>6}   {'f1-score':>8}   {'support':>7}")
    print()
    for lbl, m in sorted(per_class.items()):
        print(f"  {lbl:>{W}}   {m['precision']:>9.4f}   {m['recall']:>6.4f}"
              f"   {m['f1']:>8.4f}   {m['support']:>7}")
    print()
    print(f"  {'accuracy':>{W}}   {'':>9}   {'':>6}   {accuracy:>8.4f}   {valid:>7}")
    print(f"  {'macro avg':>{W}}   {macro_p:>9.4f}   {macro_r:>6.4f}   {macro_f:>8.4f}   {valid:>7}")
    print(f"  {'weighted avg':>{W}}   {w_p:>9.4f}   {w_r:>6.4f}   {w_f:>8.4f}   {valid:>7}")

    # Confusion Matrix
    print()
    print(f"===== Confusion Matrix =====")
    print(f"  (행=실제, 열=예측)  라벨: {labels}")
    matrix = [
        [sum(1 for t, p in zip(y_true, y_pred) if t == tl and p == pl)
         for pl in labels]
        for tl in labels
    ]
    col_w = max(max(len(str(v)) for row in matrix for v in row), 4)
    for row in matrix:
        print("  [" + "  ".join(f"{v:{col_w}}" for v in row) + "]")
    print()


print("유틸리티 함수 정의 완료")

유틸리티 함수 정의 완료


## 셀 6 · Qwen 모델 함수 정의

In [11]:
_SYSTEM_TMPL = """\

1. 너는 공정한 대한민국 법원 판사다.

2. 음주운전에 대해 형량을 판결한다. 아래 법정형과 양형기준을 참고하여 사건을 분석하고 판결을 예측한다.
 징역 선택시에는 집행유예 요건 충족여부(제51조)를 먼저 판단해서, 집행유예 요건을 충족하면 집행유예를 형종으로 선택한다.
 집행유예를 선택할 때는 징역형과 동일하게 징역형의 형량만 정할 수 있고, 벌금형은 할 수 없다.

[법정형]
{법정형}

[양형기준표]
{양형기준}



3. 입력값은 사건 분석정보로서, 범죄사실과 양형이유다.

4. 출력 답변 규칙은 다음과 같다.

- 형종은 반드시 '징역', '집행유예', '벌금' 중 하나만 출력한다.
- 형량 출력 형식은 형종에 따라 다음을 엄격히 준수한다:
  - 징역 또는 집행유예: '1년 6개월', '8개월' 등 년·개월 형식만 허용. '원' 형식 절대 불가.
  - 벌금: '700만 원' 등 원 형식만 허용. 년·개월 형식 절대 불가.
- 반드시 아래 형식으로만 답한다:
  형종: [형종]
  형량: [형량]"""


def load_model():
    print(f"[모델 로드] {MODEL_ID}  (첫 실행 시 자동 다운로드 ~6GB)")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model.eval()
    print("모델 로드 완료")
    return model, tokenizer


def predict(model, tokenizer, case_text: str, 법정형_text: str, 양형기준_text: str):
    """형종·형량 예측. 반환: (형종, 형량, 원본응답)"""
    system = _SYSTEM_TMPL.format(
        법정형=법정형_text[:1500],
        양형기준=양형기준_text[:1500],
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": f"사건 내용:\n{case_text[:1000]}"},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # 형종: 정규식 우선 → 단어 검색 폴백
    형종_m = re.search(r"형종\s*:\s*(징역|집행유예|벌금)", response)
    if 형종_m:
        형종 = 형종_m.group(1)
    else:
        # 응답 어디에든 형종 단어가 있으면 추출
        for candidate in ("징역", "집행유예", "벌금"):
            if candidate in response:
                형종 = candidate
                break
        else:
            형종 = ""

    형량_m = re.search(r"형량\s*:\s*(.+)", response)
    형량   = 형량_m.group(1).strip() if 형량_m else ""

    # 형종-형량 형식 일관성 검증
    if 형종 in ("징역", "집행유예"):
        # 원 형식이 포함되어 있으면 형량 무효 처리
        if re.search(r"[\d,]+\s*만?\s*원", 형량):
            형량 = ""
    elif 형종 == "벌금":
        # 년·개월 형식만 있고 원 형식이 없으면 형량 무효 처리
        if not re.search(r"[\d,]+\s*만?\s*원", 형량):
            형량 = ""

    return 형종, 형량, response   # 원본 응답도 함께 반환


# ── 출력 헬퍼 ─────────────────────────────────────────────────

def print_field(label: str, value: str, indent: int = 2):
    prefix = " " * indent
    if "\n" in value or len(value) > 60:
        print(f"{prefix}[{label}]")
        for line in value.splitlines():
            print(f"{prefix}  {line}")
    else:
        print(f"{prefix}[{label}] {value}")


print("모델 함수 정의 완료")

모델 함수 정의 완료


## 셀 7 · GitHub에서 파일 다운로드

> 저장소 **Public**

In [7]:
print("[1/3] GitHub에서 파일 다운로드 중...")

법정형_text   = download_docx_text(FILE_법정형)
양형기준_text = download_docx_text(FILE_양형기준)
_, rows       = download_csv_rows(FILE_판결문)

print(f"\n다운로드 완료 — 판결문 데이터: {len(rows)}건")

[1/3] GitHub에서 파일 다운로드 중...
  다운로드: https://raw.githubusercontent.com/ssc-lexdev/ssc_project/main/%EC%9D%8C%EC%A3%BC%EC%9A%B4%EC%A0%84%20%EB%B2%95%EC%A0%95%ED%98%95.docx
  다운로드: https://raw.githubusercontent.com/ssc-lexdev/ssc_project/main/%EC%9D%8C%EC%A3%BC%EC%9A%B4%EC%A0%84%20%EC%96%91%ED%98%95%EA%B8%B0%EC%A4%80%ED%91%9C.docx
  다운로드: https://raw.githubusercontent.com/ssc-lexdev/ssc_project/main/%EC%9D%8C%EC%A3%BC%EC%9A%B4%EC%A0%84_%ED%8C%90%EA%B2%B0%EB%AC%B8_%EC%B6%94%EC%B6%9C%20v5%20-%20data.csv
  인코딩 감지: cp949

다운로드 완료 — 판결문 데이터: 23건


## 셀 8 · Qwen 모델 로드

> 모델을 한 번 로드하면 셀 9(예측)를 여러 번 재실행해도 다시 로드하지 않습니다.

In [8]:
print("[2/3] Qwen 모델 로드 중...")
model, tokenizer = load_model()

[2/3] Qwen 모델 로드 중...
[모델 로드] Qwen/Qwen2.5-3B-Instruct  (첫 실행 시 자동 다운로드 ~6GB)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

모델 로드 완료


## 셀 9 · AI 예측 및 결과 출력

건별 출력 순서: **법원 → 사건 → 주문 → 범죄사실 → 양형이유 → 실제판결 → AI예측**

In [9]:
# ── 유효 데이터만 필터 (Label1이 형종인 행만 처리) ─────────────
VALID_형종 = {"징역", "집행유예", "벌금"}
valid_rows = [r for r in rows if r.get("Label1", "").strip() in VALID_형종]
print(f"전체 행: {len(rows)}  /  유효 판결 데이터: {len(valid_rows)}건\n")

y_true_종  = []   # 형종 분류성능용
y_pred_종  = []
y_true_량  = []   # 형량 분류성능용
y_pred_량  = []
비교_results = []

for i, row in enumerate(valid_rows, 1):
    actual_종  = row.get("Label1", "").strip()
    actual_량2 = row.get("Label2", "").strip()

    범죄사실  = row.get(COL_범죄사실, "").strip()
    양형이유  = row.get(COL_양형이유, "").strip()

    if not 범죄사실 and not 양형이유:
        print(f"  ⚠ [{i}] 범죄사실·양형이유 모두 비어 있음 → 건너뜀")
        continue

    case_text = f"[범죄사실]\n{범죄사실}\n\n[양형이유]\n{양형이유}"

    ai_종, ai_량_text, raw_response = predict(model, tokenizer,
                                              case_text, 법정형_text, 양형기준_text)

    if not ai_종:
        print(f"  ⚠ [{i}] 형종 파싱 실패 — 모델 원본 응답:\n    {repr(raw_response)}\n")

    # 형량 수치 비교
    actual_량_num = label2_to_numeric(actual_종, actual_량2)
    ai_량_num = (text_to_manwon(ai_량_text)
                 if ai_종 == "벌금" else text_to_months(ai_량_text))
    비교 = compare_sentences(actual_종, actual_량_num, ai_종, ai_량_num)
    비교_results.append(비교)

    # 형종 분류성능 데이터 수집
    if actual_종 and ai_종:
        y_true_종.append(actual_종)
        y_pred_종.append(ai_종)

    # 형량 분류성능 데이터 수집 (정규화된 텍스트로 비교, 허용 범위 내 동일 처리)
    actual_량_text = label2_to_text(actual_종, actual_량2)
    ai_량_norm     = normalize_형량(ai_종, ai_량_text)
    if actual_량_text and ai_량_norm:
        pred_for_report = ai_량_norm
        if actual_종 == ai_종 and actual_량_num is not None and ai_량_num is not None:
            if actual_종 in ("징역", "집행유예") and abs(actual_량_num - ai_량_num) < 12:
                pred_for_report = actual_량_text   # 1년 미만 차이 → 동일 처리
            elif actual_종 == "벌금" and abs(actual_량_num - ai_량_num) < 500:
                pred_for_report = actual_량_text   # 500만 원 미만 차이 → 동일 처리
        y_true_량.append(actual_량_text)
        y_pred_량.append(pred_for_report)

    # ── 건별 출력 ──────────────────────────────────────────────
    print(SEP)
    print(f"  ▶ [{i}/{len(valid_rows)}]")
    print_field("법원",     row.get(COL_법원, "").strip())
    print_field("사건",     row.get(COL_사건, "").strip())
    print_field("주문",     row.get(COL_주문, "").strip())
    print_field("범죄사실", 범죄사실)
    print_field("양형이유", 양형이유)
    print(SEP2)
    print("  [실제 판결문 결과]")
    print(f"    형종 : {actual_종}")
    print(f"    형량 : {actual_량_text}")
    print("  [AI 예측결과]")
    print(f"    형종 : {ai_종 if ai_종 else '(파싱 실패)'}")
    print(f"    형량 : {ai_량_text if ai_량_text else '(파싱 실패)'}")
    print(f"    비교 : {비교}")

print("\n예측 완료")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


전체 행: 23  /  유효 판결 데이터: 23건

  ▶ [1/23]
  [법원] 대구지방법원 포항지원
  [사건] 2020고단1139 도로교통법위반(음주운전), 도로교통법위반(무면
  [주문] 피고인을 징역 1년 2개월에 처한다.
  [범죄사실]
    [범죄전력] 피고인은 2008. 10. 1. 대구지방법원 포항지원에서 도로교통법위반(음주운전)죄로 벌금 100만 원, 2008. 12. 1. 같은 법원에서 도로교통법위반(음주운전)죄, 도로교통법위반(무면허운전)죄로 벌금 250만 원, 2011. 1. 13. 같은 법원에서 도로교통법위반(음주운전)죄, 도로교통법위반(무면 허운전)죄로 벌금 500만 원, 2013. 12. 18. 같은 법원에서 도로교통법위반(음주운전)죄, 도로교 통법위반(무면허운전)죄로 벌금 700만 원, 2016. 6. 30. 같은 법원에서 도로교통법위반(음주운 전)죄로 징역 6개월에 집행유예 2년을 각 선고받았고, 2019. 8. 8. 같은 법원에서 도로교통법위반 (음주운전)죄, 도로교통법위반(무면허운전)죄 등으로 징역 8개월에 집행유예 2년을 선고받고 2019. 8. 17. 그 판결이 확정되어 현재 집행유예 기간 중이다.<각주1> 【범행】 - 1 - 출처: lbox.kr/case/대구지방법원포항지원/2020고단1139 피고인은 2020. 7. 22. 01:30경 포항시 남구 B에 있는 C 호프집 앞 도로에서부터 같은 구 D에 있 는 E 앞 도로에 이르기까지 약 50m 구간에서 혈중알콜농도 0.176%의 술에 취한 상태로 자동차운 전면허를 받지 아니하고 F 렉스턴 차량을 운전하였다. 이로써 피고인은 도로교통법 제44조 제1항 또는 제2항을 2회 이상 위반함과 동시에, 자동차운전면 허를 받지 아니하고 자동차를 운전하였다.
  [양형이유]
    혈중알콜농도 수치가 매우 높고, 무면허운전까지 한 점, 피고인은 이 사건 이전에 도로교통법위반 (음주운전)죄로 6번, 도로교통법위반(무면허운전)죄로 6번 처벌받은 바 있어, 이번이 7번째 음주운 전임과 

## 셀 10 · 전체 성능 요약

In [10]:
# ① 형종 분류성능
print_report("형종 분류성능  (실제 Label1 vs AI 예측 형종)", y_true_종, y_pred_종)

# ② 형량 분류성능
print_report("형량 분류성능  (실제 형량 vs AI 예측 형량)", y_true_량, y_pred_량)

# ③ 형량 비교 분포
비교_분포 = defaultdict(int)
for v in 비교_results:
    비교_분포[v] += 1
total = len(비교_results)

print("===== 형량 비교 분포  (실제 vs AI) =====")
print(f"  {'비교결과':<26}  {'건수':>4}  {'비율':>7}")
print("-" * 45)
for k, v in sorted(비교_분포.items()):
    print(f"  {k:<26}  {v:>4}건  ({v/total*100:>5.1f}%)")
print("-" * 45)
print(f"  {'합계':<26}  {total:>4}건  (100.0%)")

valid_비교 = [v for v in 비교_results
              if v != "비교불가" and "형량 비교불가" not in v]
if valid_비교:
    print()
    ai_high = 비교_분포.get("AI판결 더 높음", 0)
    ac_high = 비교_분포.get("실제판결 더 높음", 0)
    print(f"  AI 과대추정 비율 : {ai_high / len(valid_비교) * 100:.1f}%")
    print(f"  AI 과소추정 비율 : {ac_high / len(valid_비교) * 100:.1f}%")

===== 형종 분류성능  (실제 Label1 vs AI 예측 형종) =====
                 precision   recall   f1-score   support

            벌금      0.0000   0.0000     0.0000         2
          집행유예      0.2500   1.0000     0.4000         4
            징역      1.0000   0.4118     0.5833        17

      accuracy                          0.4783        23
     macro avg      0.4167   0.4706     0.3278        23
  weighted avg      0.7826   0.4783     0.5007        23

===== Confusion Matrix =====
  (행=실제, 열=예측)  라벨: ['벌금', '집행유예', '징역']
  [   0     2     0]
  [   0     4     0]
  [   0    10     7]

===== 형량 분류성능  (실제 형량 vs AI 예측 형량) =====
                 precision   recall   f1-score   support

          10개월      0.0000   0.0000     0.0000         1
            1년      0.0000   0.0000     0.0000         7
        1년 2개월      1.0000   0.3333     0.5000         3
        1년 6개월      0.0000   0.0000     0.0000         0
            2년      0.3333   1.0000     0.5000         1
        2년 6개월      0.0000   0.0000